# nav ANN_DATE 对齐到最近交易日

任务：导入 `nav` 和 `fm`，把 `nav['ANN_DATE']` 对齐到最近 7 天内的 `fm['日期']`。

规则：
- 不修改任何原始文件，也不保存覆盖原文件。
- 以 `nav` 中已有的 `ANN_DATE` 为主，不主动补齐 `fm` 交易日。
- 每条 `ANN_DATE` 在前后 7 天内寻找最近的 `fm` 交易日。
- 如果前后都有交易日且距离相同，优先匹配前一个交易日。
- 找不到 7 天内交易日的 `nav` 记录直接删除。
- 输出变量为 `navtotradeday`；原始日期保存在 `SOURCE_ANN_DATE`，对齐后的交易日写回 `ANN_DATE`。

In [122]:
from pathlib import Path
import numpy as np
import pandas as pd

fold = Path(r"D:\BaiduNetdiskDownload\正则化基金数据\正则化基金数据")

nav_path = fold / r"基金数据\股票型_偏股混合型_nav.feather"
fm_path = fold / r"宽基指数日行情\宽基指数收益率.csv"

nav = pd.read_feather(nav_path)
fm = pd.read_csv(fm_path)
nav['PRICE_DATE'] = pd.to_datetime(nav['PRICE_DATE'].astype(str))
nav = nav[nav['PRICE_DATE']>='20020104']
print('nav shape:', nav.shape)
print('fm shape:', fm.shape)
print('nav columns:', nav.columns.tolist())
print('fm columns:', fm.columns.tolist())

nav shape: (13573177, 18)
fm shape: (5915, 7)
nav columns: ['index', 'F_INFO_WINDCODE', 'ANN_DATE', 'PRICE_DATE', 'F_NAV_UNIT', 'F_NAV_DIVACCUMULATED', 'F_NAV_ADJFACTOR', 'F_PRT_NETASSET', 'F_ASSET_MERGEDSHARESORNOT', 'NETASSET_TOTAL', 'F_NAV_ADJUSTED', 'IS_EXDIVIDENDDATE', 'F_NAV_DISTRIBUTION', 'S_INFO_ASHARECODE', 'CUM_NET_ASSET_VALUE', 'main_type', '1M', 'return']
fm columns: ['日期', '上证50dailyreturn', '中证500dailyreturn', '中证800dailyreturn', '中证1000dailyreturn', '创业板指dailyreturn', '沪深300dailyreturn']


In [124]:
# 参数与日期格式

fund_col = 'F_INFO_WINDCODE'
PRICE_DATE_col = 'PRICE_DATE'
fm_date_col = '日期'
max_match_days = 7

nav_work = nav.copy()
fm_work = fm.copy()

nav_work[PRICE_DATE_col] = pd.to_datetime(nav_work[PRICE_DATE_col].astype(str), errors='coerce')
fm_work[fm_date_col] = pd.to_datetime(fm_work[fm_date_col].astype(str), errors='coerce')

nav_work = nav_work.dropna(subset=[fund_col, PRICE_DATE_col]).copy()
fm_trading_days = pd.Index(fm_work[fm_date_col].dropna().drop_duplicates()).sort_values()

print('fm 交易日数量:', len(fm_trading_days))
print('fm 日期范围:', fm_trading_days.min(), '至', fm_trading_days.max())
print('nav PRICE_DATE 日期范围:', nav_work[PRICE_DATE_col].min(), '至', nav_work[PRICE_DATE_col].max())
print('nav 基金数量:', nav_work[fund_col].nunique())

fm 交易日数量: 5915
fm 日期范围: 2002-01-04 00:00:00 至 2026-05-27 00:00:00
nav PRICE_DATE 日期范围: 2002-01-04 00:00:00 至 2026-05-13 00:00:00
nav 基金数量: 10300


In [125]:
# 去重检查：同一基金同一 PRICE_DATE 如果有重复，保留最后一条。
# 这里只处理工作副本，不改原始文件。

dup_mask = nav_work.duplicated(subset=[fund_col, PRICE_DATE_col], keep=False)
dup_count = int(dup_mask.sum())
print('同一基金同一 PRICE_DATE 的重复行数:', dup_count)

if dup_count > 0:
    display(nav_work.loc[dup_mask, [fund_col, PRICE_DATE_col]].head(20))

nav_unique = (
    nav_work
    .sort_values([fund_col, PRICE_DATE_col])
    .drop_duplicates(subset=[fund_col, PRICE_DATE_col], keep='last')
    .copy()
)

同一基金同一 PRICE_DATE 的重复行数: 0


In [126]:
# 不做基金内部 PRICE_DATE 连续性的前置检查。
# 连续性只在完成 PRICE_DATE -> 最近 fm 交易日匹配后，
# 根据匹配日期差是否超过 max_match_days 来统计判断。

print('跳过基金内部 PRICE_DATE gap 前置检查。')

跳过基金内部 PRICE_DATE gap 前置检查。


In [127]:
# 核心对齐：先把每条 SOURCE_PRICE_DATE 匹配到最近的 fm 交易日。
# 然后统计 DATE_SHIFT_DAYS 是否超过 max_match_days；超过的记录从最终数据中删除。

nav_match = nav_unique.copy()
nav_match['SOURCE_PRICE_DATE'] = nav_match[PRICE_DATE_col]

fm_values = fm_trading_days.to_numpy(dtype='datetime64[ns]')
src_values = nav_match['SOURCE_PRICE_DATE'].to_numpy(dtype='datetime64[ns]')

pos_next = np.searchsorted(fm_values, src_values, side='left')
pos_prev = pos_next - 1

has_prev = pos_prev >= 0
has_next = pos_next < len(fm_values)

prev_dates = np.full(len(src_values), np.datetime64('NaT'), dtype='datetime64[ns]')
next_dates = np.full(len(src_values), np.datetime64('NaT'), dtype='datetime64[ns]')
prev_dates[has_prev] = fm_values[pos_prev[has_prev]]
next_dates[has_next] = fm_values[pos_next[has_next]]

one_day = np.timedelta64(1, 'D')
big_days = np.iinfo('int32').max

prev_gap_days = np.full(len(src_values), big_days, dtype='int32')
next_gap_days = np.full(len(src_values), big_days, dtype='int32')
prev_gap_days[has_prev] = ((src_values[has_prev] - prev_dates[has_prev]) / one_day).astype('int32')
next_gap_days[has_next] = ((next_dates[has_next] - src_values[has_next]) / one_day).astype('int32')

# 距离相同的时候使用前一个交易日，所以这里用 <=。
use_prev = prev_gap_days <= next_gap_days
best_gap_days = np.minimum(prev_gap_days, next_gap_days)
matched_dates = np.where(use_prev, prev_dates, next_dates)

has_any_fm_match = best_gap_days < big_days
over_match_limit = has_any_fm_match & (best_gap_days > max_match_days)
matched = has_any_fm_match & (best_gap_days <= max_match_days)

matched_all = nav_match.loc[has_any_fm_match].copy()
matched_all['MATCHED_FM_DATE'] = matched_dates[has_any_fm_match]
matched_all['MATCHED_PRICE_DATE'] = matched_all['MATCHED_FM_DATE']
matched_all['DATE_SHIFT_DAYS'] = best_gap_days[has_any_fm_match]

over_match_limit_rows = matched_all.loc[matched_all['DATE_SHIFT_DAYS'] > max_match_days, [fund_col, 'SOURCE_PRICE_DATE', 'MATCHED_FM_DATE', 'DATE_SHIFT_DAYS']].copy()
no_fm_match_rows = nav_match.loc[~has_any_fm_match, [fund_col, 'SOURCE_PRICE_DATE']].copy()
unmatched_rows = pd.concat([
    over_match_limit_rows[[fund_col, 'SOURCE_PRICE_DATE']],
    no_fm_match_rows[[fund_col, 'SOURCE_PRICE_DATE']],
], ignore_index=True)

match_gap_summary = (
    matched_all
    .groupby(fund_col)['DATE_SHIFT_DAYS']
    .agg(
        matched_rows_before_5d_filter='size',
        max_shift_days='max',
        over_5d_count=lambda s: int((s > max_match_days).sum()),
    )
    .reset_index()
    .sort_values(['over_5d_count', 'max_shift_days'], ascending=False)
)
match_gap_over_limit_summary = match_gap_summary.loc[match_gap_summary['over_5d_count'] > 0].copy()

navtotradeday = nav_match.loc[matched].copy()
navtotradeday['MATCHED_FM_DATE'] = matched_dates[matched]
navtotradeday[PRICE_DATE_col] = navtotradeday['MATCHED_FM_DATE']
navtotradeday['DATE_SHIFT_DAYS'] = best_gap_days[matched]

print('对齐前 nav_unique 行数:', len(nav_unique))
print('能匹配到最近 fm 交易日的行数:', int(has_any_fm_match.sum()))
print('匹配日期差超过', max_match_days, '天的行数:', len(over_match_limit_rows))
print('完全找不到 fm 交易日的行数:', len(no_fm_match_rows))
print('最终保留行数:', len(navtotradeday))
print('匹配日期差超过', max_match_days, '天的基金数量:', over_match_limit_rows[fund_col].nunique())
print('对齐后基金数量:', navtotradeday[fund_col].nunique())

print('\n匹配日期差超过 max_match_days 的基金汇总 Top 30:')
display(match_gap_over_limit_summary.head(30))

print('\n匹配日期差超过 max_match_days 的明细示例:')
display(over_match_limit_rows.head(30))

display(navtotradeday[[fund_col, PRICE_DATE_col, 'SOURCE_PRICE_DATE', 'DATE_SHIFT_DAYS']].head())

对齐前 nav_unique 行数: 13573177
能匹配到最近 fm 交易日的行数: 13573177
匹配日期差超过 7 天的行数: 0
完全找不到 fm 交易日的行数: 0
最终保留行数: 13573177
匹配日期差超过 7 天的基金数量: 0
对齐后基金数量: 10300

匹配日期差超过 max_match_days 的基金汇总 Top 30:


,F_INFO_WINDCODE,matched_rows_before_5d_filter,max_shift_days,over_5d_count



匹配日期差超过 max_match_days 的明细示例:


,F_INFO_WINDCODE,SOURCE_PRICE_DATE,MATCHED_FM_DATE,DATE_SHIFT_DAYS


,F_INFO_WINDCODE,PRICE_DATE,SOURCE_PRICE_DATE,DATE_SHIFT_DAYS
4,000001.OF,2002-01-04,2002-01-04,0
5,000001.OF,2002-01-11,2002-01-11,0
6,000001.OF,2002-01-18,2002-01-18,0
7,000001.OF,2002-01-25,2002-01-25,0
8,000001.OF,2002-01-30,2002-01-30,0


In [128]:
# 最终处理与结果检查

# 1. 对齐后如果出现同一基金同一交易日重复，保留缺失值更少的那一行。
rows_before_dedup = len(navtotradeday)
dup_before = navtotradeday.duplicated(subset=[fund_col, PRICE_DATE_col], keep=False)
duplicate_rows_before_drop = navtotradeday.loc[dup_before].copy()

navtotradeday['_NA_COUNT'] = navtotradeday.isna().sum(axis=1)
navtotradeday = (
    navtotradeday
    .sort_values([fund_col, PRICE_DATE_col, '_NA_COUNT', 'DATE_SHIFT_DAYS', 'SOURCE_PRICE_DATE'])
    .drop_duplicates(subset=[fund_col, PRICE_DATE_col], keep='first')
    .drop(columns=['_NA_COUNT'])
    .copy()
)
rows_after_dedup = len(navtotradeday)
dup_after = navtotradeday.duplicated(subset=[fund_col, PRICE_DATE_col], keep=False)

print('去重前重复行数:', int(dup_before.sum()))
print('去重后重复行数:', int(dup_after.sum()))
print('因重复日期被删除的行数:', rows_before_dedup - rows_after_dedup)

if len(duplicate_rows_before_drop) > 0:
    print('\n重复日期删除前示例:')
    display(duplicate_rows_before_drop[[fund_col, PRICE_DATE_col, 'SOURCE_PRICE_DATE', 'DATE_SHIFT_DAYS']].head(30))

# 2. 检查 nav 原始 PRICE_DATE 与匹配到的 fm 交易日差多少天。
bad_not_fm = navtotradeday.loc[~navtotradeday[PRICE_DATE_col].isin(fm_trading_days)]
bad_shift = navtotradeday.loc[navtotradeday['DATE_SHIFT_DAYS'].gt(max_match_days) | navtotradeday['DATE_SHIFT_DAYS'].lt(0)]

date_shift_summary = (
    navtotradeday
    .groupby(fund_col)['DATE_SHIFT_DAYS']
    .agg(row_count='size', mean_shift_days='mean', max_shift_days='max', shifted_rows=lambda s: int((s > 0).sum()))
    .reset_index()
    .sort_values(['max_shift_days', 'shifted_rows'], ascending=False)
)

print('\n输出 PRICE_DATE 不属于 fm 日期的行数:', len(bad_not_fm))
print('DATE_SHIFT_DAYS 不在 0 到 max_match_days 之间的行数:', len(bad_shift))
print('\nDATE_SHIFT_DAYS 分布:')
display(navtotradeday['DATE_SHIFT_DAYS'].value_counts().sort_index().rename('count').to_frame())
print('\n每只基金日期偏移汇总 Top 30:')
display(date_shift_summary.head(30))

# 3. 检查基金存续期间内，是否存在 fm 有日期但 navtotradeday 没有日期。
# 存续期间用去重后的 navtotradeday 中每只基金的最早/最晚交易日定义。
fund_life_after_match = (
    navtotradeday
    .groupby(fund_col)[PRICE_DATE_col]
    .agg(fund_start='min', fund_end='max')
    .reset_index()
)

fm_values = fm_trading_days.to_numpy(dtype='datetime64[ns]')
start_pos = np.searchsorted(fm_values, fund_life_after_match['fund_start'].to_numpy(dtype='datetime64[ns]'), side='left')
end_pos = np.searchsorted(fm_values, fund_life_after_match['fund_end'].to_numpy(dtype='datetime64[ns]'), side='right')
counts = end_pos - start_pos
valid = counts > 0

expected_funds = np.repeat(fund_life_after_match.loc[valid, fund_col].to_numpy(), counts[valid])
expected_dates = np.concatenate([
    fm_values[s:e] for s, e in zip(start_pos[valid], end_pos[valid])
]) if valid.any() else np.array([], dtype='datetime64[ns]')
expected_fund_dates = pd.DataFrame({fund_col: expected_funds, PRICE_DATE_col: expected_dates})

actual_fund_dates = navtotradeday[[fund_col, PRICE_DATE_col]].drop_duplicates()
missing_fm_dates = expected_fund_dates.merge(
    actual_fund_dates,
    on=[fund_col, PRICE_DATE_col],
    how='left',
    indicator=True,
).loc[lambda x: x['_merge'].eq('left_only'), [fund_col, PRICE_DATE_col]]

missing_fm_summary = (
    missing_fm_dates
    .groupby(fund_col)[PRICE_DATE_col]
    .agg(missing_fm_date_count='size', first_missing_fm_date='min', last_missing_fm_date='max')
    .reset_index()
    .sort_values('missing_fm_date_count', ascending=False)
)

print('\n基金存续期间 fm 有但 navtotradeday 没有的 基金-日期 数量:', len(missing_fm_dates))
print('存在缺失 fm 日期的基金数量:', len(missing_fm_summary))
print('\n缺失 fm 日期基金汇总 Top 30:')
display(missing_fm_summary.head(30))
print('\n缺失 fm 日期明细示例:')
display(missing_fm_dates.head(30))

print('\n无法匹配而删除的原始 nav 记录示例:')
display(unmatched_rows.head(30))

去重前重复行数: 137756
去重后重复行数: 0
因重复日期被删除的行数: 68984

重复日期删除前示例:


,F_INFO_WINDCODE,PRICE_DATE,SOURCE_PRICE_DATE,DATE_SHIFT_DAYS
53,000001.OF,2002-04-01,2002-03-31,1
41,000001.OF,2002-04-01,2002-04-01,0
144,000001.OF,2002-07-01,2002-06-30,1
102,000001.OF,2002-07-01,2002-07-01,0
167,000001.OF,2002-09-27,2002-09-27,0
180,000001.OF,2002-09-27,2002-09-30,3
955,000001.OF,2005-12-30,2005-12-30,0
956,000001.OF,2005-12-30,2005-12-31,1
1137,000001.OF,2006-09-29,2006-09-29,0
1152,000001.OF,2006-09-29,2006-09-30,1



输出 PRICE_DATE 不属于 fm 日期的行数: 0
DATE_SHIFT_DAYS 不在 0 到 max_match_days 之间的行数: 0

DATE_SHIFT_DAYS 分布:


,count
DATE_SHIFT_DAYS,
0,13439166
1,40575
2,24208
3,4
4,240



每只基金日期偏移汇总 Top 30:


...

,F_INFO_WINDCODE,missing_fm_date_count,first_missing_fm_date,last_missing_fm_date
7477,184722.SZ,2859,2002-07-08,2017-06-29
7476,184721.SZ,2852,2002-03-25,2017-03-16
7478,184728.SZ,2847,2002-01-07,2016-12-29
7680,500056.SH,2818,2002-03-13,2016-12-29
7678,500038.SH,2770,2002-01-07,2016-08-04
7669,500015.SH,2455,2002-01-07,2014-12-10
7461,184699.SZ,2433,2002-01-07,2014-11-03
7667,500011.SH,2408,2002-01-07,2014-09-11
7460,184698.SZ,2366,2002-01-07,2014-06-26
7457,184692.SZ,2352,2002-01-07,2014-05-29



缺失 fm 日期明细示例:


,F_INFO_WINDCODE,PRICE_DATE
1,000001.OF,2002-01-07
2,000001.OF,2002-01-08
3,000001.OF,2002-01-09
4,000001.OF,2002-01-10
6,000001.OF,2002-01-14
7,000001.OF,2002-01-15
8,000001.OF,2002-01-16
9,000001.OF,2002-01-17
11,000001.OF,2002-01-21
12,000001.OF,2002-01-22



无法匹配而删除的原始 nav 记录示例:


,F_INFO_WINDCODE,SOURCE_PRICE_DATE


In [129]:
# # 检查调整后的 PRICE_DATE 序列相邻 gap 大小和数量。
# # 这里检查的是最终 navtotradeday 中，每只基金按对齐后的交易日排序后的相邻日期差。

# adjusted_gap = navtotradeday[[fund_col, PRICE_DATE_col]].drop_duplicates().sort_values([fund_col, PRICE_DATE_col]).copy()
# adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'] = adjusted_gap.groupby(fund_col)[PRICE_DATE_col].diff().dt.days

# adjusted_gap_summary = (
#     adjusted_gap
#     .groupby(fund_col)['ADJUSTED_PRICE_DATE_GAP_DAYS']
#     .agg(
#         date_count='size',
#         max_adjusted_gap_days='max',
#         mean_adjusted_gap_days='mean',
#         gap_gt_1d_count=lambda s: int((s > 1).sum()),
#         gap_gt_3d_count=lambda s: int((s > 3).sum()),
#         gap_gt_5d_count=lambda s: int((s > 5).sum()),
#         gap_gt_10d_count=lambda s: int((s > 10).sum()),
#         gap_gt_30d_count=lambda s: int((s > 30).sum()),
#     )
#     .reset_index()
#     .sort_values(['max_adjusted_gap_days', 'gap_gt_10d_count'], ascending=False)
# )

# adjusted_large_gap_rows = adjusted_gap.loc[adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'] > 15].copy()
# adjusted_large_gap_summary = (
#     adjusted_large_gap_rows
#     .groupby(fund_col)
#     .agg(
#         large_gap_count=('ADJUSTED_PRICE_DATE_GAP_DAYS', 'size'),
#         max_large_gap_days=('ADJUSTED_PRICE_DATE_GAP_DAYS', 'max'),
#         first_large_gap_end_date=(PRICE_DATE_col, 'min'),
#         last_large_gap_end_date=(PRICE_DATE_col, 'max'),
#     )
#     .reset_index()
#     .sort_values(['large_gap_count', 'max_large_gap_days'], ascending=False)
# )

# print('调整后基金-日期总数:', len(adjusted_gap))
# print('调整后相邻日期 gap > 1 天的总次数:', int((adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'] > 1).sum()))
# print('调整后相邻日期 gap > 3 天的总次数:', int((adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'] > 3).sum()))
# print('调整后相邻日期 gap > 5 天的总次数:', int((adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'] > 5).sum()))
# print('调整后相邻日期 gap > 15 天的总次数:', int((adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'] > 15).sum()))
# print('调整后存在 gap > 15 的基金数量:', adjusted_large_gap_rows[fund_col].nunique())

# print('\n调整后 gap 天数分布 Top 30:')
# display(adjusted_gap['ADJUSTED_PRICE_DATE_GAP_DAYS'].value_counts().sort_index().rename('count').to_frame().tail(30))

# print('\n每只基金调整后 gap 汇总 Top 30:')
# display(adjusted_gap_summary.head(30))

# print('\n调整后 gap > max_match_days 的明细示例:')
# display(adjusted_large_gap_rows.head(30))

# print('\n调整后 gap > max_match_days 的基金汇总 Top 30:')
# display(adjusted_large_gap_summary.head(30))

In [130]:
# =========================
# 交易日历
# =========================

trade_days = (
    pd.Series(pd.to_datetime(fm['日期']).dropna().unique())
    .sort_values()
    .reset_index(drop=True)
)

trade_day_index = pd.DataFrame({
    'TRADE_DT': trade_days,
    'TRADE_DAY_NUM': range(len(trade_days))
})

# =========================
# 基金日期序列
# =========================

adjusted_gap = (
    navtotradeday[[fund_col, PRICE_DATE_col]]
    .drop_duplicates()
    .copy()
)

adjusted_gap[PRICE_DATE_col] = pd.to_datetime(
    adjusted_gap[PRICE_DATE_col]
)

adjusted_gap = adjusted_gap.sort_values(
    [fund_col, PRICE_DATE_col]
)

# =========================
# 合并交易日编号
# =========================

adjusted_gap = adjusted_gap.merge(
    trade_day_index,
    left_on=PRICE_DATE_col,
    right_on='TRADE_DT',
    how='left'
)

# =========================
# 前一个交易日编号
# =========================

adjusted_gap['PREV_TRADE_DAY_NUM'] = (
    adjusted_gap
    .groupby(fund_col)['TRADE_DAY_NUM']
    .shift(1)
)

# =========================
# 缺失交易日数
# =========================

adjusted_gap['MISSING_TRADE_DAYS'] = (
    adjusted_gap['TRADE_DAY_NUM']
    - adjusted_gap['PREV_TRADE_DAY_NUM']
    - 1
)

# =========================
# 是否超大 gap
# =========================

adjusted_gap['IS_LARGE_GAP'] = (
    adjusted_gap['MISSING_TRADE_DAYS']
    > max_match_days
)

# =========================
# 超大 gap 明细
# =========================

large_gap_rows = adjusted_gap[
    adjusted_gap['IS_LARGE_GAP']
].copy()

# =========================
# 存在超大 gap 的基金
# =========================

large_gap_funds = (
    large_gap_rows[fund_col]
    .drop_duplicates()
)

# =========================
# 每只基金汇总
# =========================

large_gap_summary = (
    large_gap_rows
    .groupby(fund_col)
    .agg(
        large_gap_count=('IS_LARGE_GAP', 'sum'),
        max_missing_trade_days=('MISSING_TRADE_DAYS', 'max'),
        first_large_gap_date=(PRICE_DATE_col, 'min'),
        last_large_gap_date=(PRICE_DATE_col, 'max'),
    )
    .reset_index()
    .sort_values(
        ['large_gap_count', 'max_missing_trade_days'],
        ascending=False
    )
)

# =========================
# 输出
# =========================

print('基金总数:', adjusted_gap[fund_col].nunique())

print('存在超大 gap 的基金数量:', len(large_gap_funds))

print('超大 gap 总次数:', len(large_gap_rows))

print('\n缺失交易日分布 Top 30:')
display(
    adjusted_gap['MISSING_TRADE_DAYS']
    .value_counts()
    .sort_index()
    .tail(30)
)

print('\n超大 gap 基金汇总 Top 30:')
display(large_gap_summary.head(30))

print('\n超大 gap 明细 Top 30:')
display(
    large_gap_rows
    .sort_values('MISSING_TRADE_DAYS', ascending=False)
    .head(30)
)

基金总数: 10300
存在超大 gap 的基金数量: 115
超大 gap 总次数: 134

缺失交易日分布 Top 30:


MISSING_TRADE_DAYS
52.0      1
53.0      1
54.0      2
55.0      1
59.0      1
60.0      1
78.0      1
83.0      1
86.0      1
92.0      1
93.0      2
99.0      4
100.0     2
105.0     1
108.0     1
119.0     1
130.0     1
148.0     1
219.0     1
236.0     1
242.0     1
564.0     1
565.0     1
580.0     1
713.0     1
810.0     1
835.0     1
973.0     1
1184.0    1
1266.0    1
Name: count, dtype: int64


超大 gap 基金汇总 Top 30:


,F_INFO_WINDCODE,large_gap_count,max_missing_trade_days,first_large_gap_date,last_large_gap_date
45,004049.OF,4,835.0,2020-08-11,2022-08-11
27,002872.OF,4,580.0,2018-12-26,2019-12-24
58,005950.OF,3,564.0,2018-09-28,2021-04-30
40,003781.OF,3,92.0,2018-02-02,2018-08-28
64,007334.OF,3,43.0,2020-03-31,2020-06-30
25,002834.OF,2,1184.0,2018-03-30,2023-02-17
26,002839.OF,2,973.0,2020-08-11,2022-02-08
8,001523.OF,2,713.0,2019-02-18,2019-05-08
47,004675.OF,2,236.0,2018-03-30,2019-03-22
19,002410.OF,2,78.0,2016-04-05,2017-07-13



超大 gap 明细 Top 30:


,F_INFO_WINDCODE,PRICE_DATE,TRADE_DT,TRADE_DAY_NUM,PREV_TRADE_DAY_NUM,MISSING_TRADE_DAYS,IS_LARGE_GAP
3594518,004051.OF,2022-08-11,2022-08-11,4999,3732.0,1266.0,True
3005486,002834.OF,2023-02-17,2023-02-17,5123,3938.0,1184.0,True
3011667,002839.OF,2020-08-11,2020-08-11,4513,3539.0,973.0,True
3591058,004049.OF,2020-08-11,2020-08-11,4513,3677.0,835.0,True
2622752,002390.OF,2019-06-17,2019-06-17,4231,3420.0,810.0,True
1514761,001523.OF,2019-02-18,2019-02-18,4151,3437.0,713.0,True
3034445,002872.OF,2018-12-26,2018-12-26,4120,3539.0,580.0,True
2473054,002232.OF,2020-09-29,2020-09-29,4548,3982.0,565.0,True
4728344,005950.OF,2021-04-30,2021-04-30,4688,4123.0,564.0,True
6694613,010669.OF,2023-07-25,2023-07-25,5229,4986.0,242.0,True


In [131]:
# 保存匹配后的数据。
# 不覆盖原始 nav 文件，只生成新的 feather 文件。
navtotradeday = navtotradeday[~navtotradeday['F_INFO_WINDCODE'].isin(large_gap_funds)]
output_path = fold / r"基金数据\交易日偏股型基金.feather"
navtotradeday.reset_index().to_feather(output_path)

print('已保存:', output_path)
print('保存行数:', len(navtotradeday))
print('保存基金数量:', navtotradeday[fund_col].nunique())

已保存: D:\BaiduNetdiskDownload\正则化基金数据\正则化基金数据\基金数据\交易日偏股型基金.feather
保存行数: 13269861
保存基金数量: 10185
